# Chroma DB ingestion and Q&A

## What this notebook does

This notebook introduces the core mechanics of Retrieval-Augmented Generation (RAG) with ChromaDB. It starts by creating an in-memory Chroma collection, inserting short text chunks about Paestum, and running semantic searches to retrieve the chunks that best match a question.

After the retrieval examples, the notebook builds a simple RAG flow from scratch: retrieve context from the vector database, insert that context into a prompt, and send the prompt to an LLM. The LLM provider can be selected at the beginning of the notebook, so the same workflow can run with OpenAI, Ollama, or Gemini.


In [7]:
# Select the LLM provider for the notebook: "openai", "ollama", or "gemini".
# You can also set these values in the environment before starting Jupyter.
import os
import getpass
from openai import OpenAI
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")

SUPPORTED_LLM_PROVIDERS = {"openai", "ollama", "gemini"}
if LLM_PROVIDER not in SUPPORTED_LLM_PROVIDERS:
    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}. Use one of: {sorted(SUPPORTED_LLM_PROVIDERS)}")


def _get_api_key(*names):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return getpass.getpass(f"Enter your {names[0]}")


def build_llm(provider):
    if provider == "openai":
        return OpenAI(api_key=_get_api_key("OPENAI_API_KEY")), OPENAI_MODEL
    if provider == "ollama":
        return ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL), OLLAMA_MODEL
    if provider == "gemini":
        api_key = _get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY")
        return ChatGoogleGenerativeAI(model=GEMINI_MODEL, google_api_key=api_key), GEMINI_MODEL


llm_client, ACTIVE_LLM_MODEL = build_llm(LLM_PROVIDER)
print(f"Using {LLM_PROVIDER} with model {ACTIVE_LLM_MODEL}")


Using gemini with model gemini-flash-latest


## Ingestion

In [8]:
import chromadb
chroma_client = chromadb.Client()

In [9]:
tourism_collection = chroma_client.create_collection(
    name="tourism_collection")

InternalError: Collection [tourism_collection] already exists

### Inserting the content

The collection stores the Paestum text as separate document chunks. Each chunk is small enough to be searched independently, which is the usual ingestion pattern in a RAG pipeline: split the source text, store the chunks, and let the vector store create embeddings for them.

In this example, Chroma uses its default embedding function. The `metadatas` entries keep track of the source URL for each chunk, while the explicit `ids` make each stored document addressable. After this step, the content is no longer just text in the notebook: it has been indexed in the vector store and can be queried semantically.


In [10]:
tourism_collection.add(
    documents=[
        "Paestum, Greek Poseidonia, ancient city in southern Italy near the west coast, 22 miles (35 km) southeast of modern Salerno and 5 miles (8 km) south of the Sele (ancient Silarus) River. Paestum is noted for its splendidly preserved Greek temples.", 
        "Poseidonia was probably founded about 600 BC by Greek colonists from Sybaris, along the Gulf of Taranto, and it had become a flourishing town by 540, judging from its temples. After many years’ resistance the city came under the domination of the Lucanians (an indigenous Italic people) sometime before 400 BC, after which its name was changed to Paestum. Alexander, the king of Epirus, defeated the Lucanians at Paestum about 332 BC, but the city remained Lucanian until 273, when it came under Roman rule and a Latin colony was founded there. The city supported Rome during the Second Punic War. The locality was still prosperous during the early years of the Roman Empire, but the gradual silting up of the mouth of the Silarus River eventually created a malarial swamp, and Paestum was finally deserted after being sacked by Muslim raiders in AD 871. The abandoned site’s remains were rediscovered in the 18th century.",
        "The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancient Lucanian tomb that contained Greek frescoes painted in the early classical style. Paestum’s archaeological museum contains these and other treasures from the site."
    ],
    metadatas=[
        {"source": "https://www.britannica.com/place/Paestum"}, 
        {"source": "https://www.britannica.com/place/Paestum"},
        {"source": "https://www.britannica.com/place/Paestum"}
    ],
    ids=["paestum-br-01", "paestum-br-02", "paestum-br-03"]
)

## Q&A

### Performing a semantic search

A semantic search query sends only the question to Chroma. Unlike a direct prompt to an LLM, the full Paestum text does not need to be included in the request because it has already been inserted into the collection.

Chroma embeds the question, compares it with the stored document embeddings, and returns the closest chunk. With `n_results=1`, the query asks for only the best match. The result includes the matching document, its metadata, its id, and a distance score that indicates how close the query embedding is to the document embedding.


In [11]:
results = tourism_collection.query(
    query_texts=["How many Doric temples are in Paestum"],
    n_results=1
)
print(results)

{'ids': [['paestum-br-03']], 'embeddings': None, 'documents': [['The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancien

### Checking semantic proximity

Requesting more results makes the ranking visible. With `n_results=3`, Chroma returns all three Paestum chunks ordered by semantic proximity to the question.

The chunk about the Greek sacred areas and the three Doric temples should appear first, because its embedding is closest to the question embedding. Lower-ranked chunks are still related to Paestum, but they are less directly relevant to the specific question. The vector database is not generating an answer here; it is retrieving the text chunks that an LLM can later use as context.


In [12]:
results = tourism_collection.query(
    query_texts=["How many Doric temples are in Paestum"],
    n_results=3
)
print(results)

{'ids': [['paestum-br-03', 'paestum-br-01', 'paestum-br-02']], 'embeddings': None, 'documents': [['The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July

# RAG from scratch

## 6.3.1 Retrieving content from the vector database

The first step in the RAG flow is retrieval. The vector store already contains the Paestum chunks, so the chatbot does not need to send the whole source text to the LLM. It only needs to ask Chroma for the chunk that is closest to the user's question.

The `query_vector_database()` function wraps this search into a reusable operation. It sends the question to Chroma, asks for one result, extracts the first returned document, and gives that text back to the rest of the pipeline as context.


In [13]:
def query_vector_database(question):
    results = tourism_collection.query(
    query_texts=[question],
    n_results=1)

    results_text = results['documents'][0][0]

    return results_text

### Testing the retriever

This test checks that the retriever finds the expected Paestum chunk for the question about Doric temples. The returned text should contain the passage that mentions the three Doric temples, confirming that semantic search found the relevant context before any LLM is involved.


In [14]:
results_text = query_vector_database("How many Doric temples are in Paestum")
print(results_text)

The ancient Greek part of Paestum consists of two sacred areas containing three Doric temples in a remarkable state of preservation. During the ensuing Roman period a typical forum and town layout grew up between the two ancient Greek sanctuaries. Of the three temples, the Temple of Athena (the so-called Temple of Ceres) and the Temple of Hera I (the so-called Basilica) date from the 6th century BC, while the Temple of Hera II (the so-called Temple of Neptune) was probably built about 460 BC and is the best preserved of the three. The Temple of Peace in the forum is a Corinthian-Doric building begun perhaps in the 2nd century BC. Traces of a Roman amphitheatre and other buildings, as well as intersecting main streets, have also been found. The circuit of the town walls, which are built of travertine blocks and are 15–20 feet (5–6 m) thick, is about 3 miles (5 km) in circumference. In July 1969 a farmer uncovered an ancient Lucanian tomb that contained Greek frescoes painted in the earl

## 6.3.2 Invoking the LLM

After retrieval, the next step is generation. The LLM receives two pieces of information: the original question and the retrieved context. The prompt template is responsible for combining them into a single instruction.

The `execute_llm_prompt()` function sends that prompt to the selected model. In this notebook the provider can be OpenAI, Ollama, or Gemini, but the surrounding RAG flow stays the same: retrieve context, build a prompt, invoke the model, and return the model's answer.


In [15]:
def prompt_template(question, text):
    return f'Read the following text and answer this question: {question}. \nText: {text}'

In [16]:
SYSTEM_PROMPT = "You are an assistant for question-answering tasks."


def _message_text(response):
    if isinstance(response, str):
        return response
    if hasattr(response, "content"):
        content = response.content
        if isinstance(content, str):
            return content
        if isinstance(content, list):
            parts = []
            for item in content:
                if isinstance(item, dict) and "text" in item:
                    parts.append(str(item["text"]))
                else:
                    parts.append(str(item))
            return "\n".join(parts)
    return str(response)


def execute_llm_prompt(prompt_input):
    if LLM_PROVIDER == "openai":
        prompt_response = llm_client.chat.completions.create(
            model=ACTIVE_LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt_input},
            ],
        )
        return prompt_response.choices[0].message.content

    prompt_response = llm_client.invoke([
        ("system", SYSTEM_PROMPT),
        ("user", prompt_input),
    ])
    return _message_text(prompt_response)


### Using a simple Q&A prompt

The first prompt is intentionally simple: it asks the model to read the retrieved text and answer the question. This is enough to demonstrate the mechanics of RAG, but it is not the safest possible instruction.

The test question asks for the total number of columns in the three temples. The retrieved context talks about the temples but does not include their column counts. This makes the example useful for observing whether the model stays grounded in the retrieved context or tries to infer missing facts.


In [17]:
trick_question = "How many columns have the three temples got in total?"
tq_result_text = query_vector_database(trick_question)
tq_prompt = prompt_template(trick_question , tq_result_text)
tq_prompt_response = execute_llm_prompt(tq_prompt)
print(tq_prompt_response)

Based on the text provided, it is **not possible to answer this question**. The text mentions the names, styles, and approximate dates of the three temples, but it does not provide any information regarding the number of columns they have.


## Safer prompt implementation

A safer RAG prompt tells the model to answer only from the retrieved context and to say that it does not know when the answer is missing. This reduces hallucination risk because the model is explicitly constrained to the information returned by the vector database.

The revised prompt also keeps the answer concise. That matters in Q&A systems because the goal is not to summarize everything in the retrieved chunk, but to answer the user's specific question with the available evidence.


In [19]:
def prompt_template(question, text):
    return f'Use the following pieces of retrieved context to answer the question. Only use the retrieved context to answer the question. If you don\'t know the answer, or the answer is not contained in the retrieved context, just say that you don\'t know. Use three sentences maximum and keep the answer concise. \nQuestion: {question}\nContext: {text}. Remember: if you do not know, just say: I do not know. Do not make up an answer. For example do not say the three temples have got a total of three columns. \nAnswer:'

### Testing the safer prompt

The same trick question is submitted again after changing the prompt. Since the retrieved context does not contain the total number of columns, the desired behavior is for the model to refuse to invent an answer and respond that it does not know.


In [21]:
trick_question = "How many columns have the three temples got in total?"
tq_result_text = query_vector_database(trick_question)
tq_prompt = prompt_template(trick_question , tq_result_text)
tq_prompt_response = execute_llm_prompt(tq_prompt)
print(tq_prompt_response)

I do not know.


## Building a chatbot

The chatbot function combines the three operations into one RAG pipeline. It retrieves the most relevant context from Chroma, inserts that context into the prompt, and invokes the selected LLM to synthesize the final answer.

This small function captures the basic architecture used by larger Q&A systems. Frameworks such as LangChain or LlamaIndex can hide these steps behind abstractions, but the underlying flow is the same: question, retrieval, prompt construction, generation.


In [23]:
def my_chatbot(question):
    results_text = query_vector_database(question) #A    
    prompt_input = prompt_template(question, 
                                   results_text) #B
    prompt_output = execute_llm_prompt(
        prompt_input) #C

    return prompt_output
#A Retrieve content from vector store
#B Create LLM prompt
#C Execute LLM prompt

### Testing the chatbot

The final test asks a multi-part question about the Paestum temples. The chatbot should answer what is supported by the retrieved context and avoid filling gaps with unsupported information. This demonstrates why RAG is useful: it narrows the LLM's attention to retrieved evidence instead of relying only on the model's internal knowledge.


In [24]:
question = """Let me know how many temples there
are in Paestum, who constructed them, and what 
architectural style they are"""
result = my_chatbot(question)
print(result)

Paestum contains four temples: three Doric temples (Athena, Hera I, and Hera II) and the Corinthian-Doric Temple of Peace. The three Doric temples were constructed during the ancient Greek period, while the Temple of Peace was built during the Roman period.


## 6.3.4 Recap of RAG terminology

The notebook has now implemented the core RAG loop from scratch. Ingestion stores text chunks and embeddings in Chroma. Retrieval finds the chunks closest to a user question. Generation uses those retrieved chunks as context for the LLM answer.

The important design point is that retrieval and generation are separate responsibilities. Chroma finds relevant text; the LLM turns the question and retrieved context into a natural-language answer. Keeping those roles clear makes RAG systems easier to debug and improve.
